<a href="https://colab.research.google.com/github/hagalaz-star/selfcheckgpt-reproduction/blob/main/selfcheckgpt_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q selfcheckgpt datasets

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
device: cpu
torch: 2.14.1+cpu


In [2]:
from datasets import load_dataset

ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
print(ds)                      # split 이름, 행 개수, 컬럼 목록

split_name = list(ds.keys())[0]
data = ds[split_name]
print("\n사용할 split:", split_name)
print("\n[컬럼 타입]")
for name, feat in data.features.items():
    print(f"- {name}: {feat}")

print("\n[첫 번째 샘플]")
ex = data[0]
for k, v in ex.items():
    if isinstance(v, list):
        print(f"\n■ {k} | list, 길이 {len(v)}")
        for item in v[:3]:
            print("   ·", str(item)[:150])
    else:
        print(f"\n■ {k} | {type(v).__name__}")
        print("   ", str(v)[:300])

README.md:   0%|          | 0.00/2.45k [00:00<?, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…): reconstructing file:   0%|          |  0.00B / 2.56MB            

data/evaluation-00000-of-00001-e91191b8f(…): downloading bytes:           |  0.00B            

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

DatasetDict({
    evaluation: Dataset({
        features: ['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'],
        num_rows: 238
    })
})

사용할 split: evaluation

[컬럼 타입]
- gpt3_text: Value('string')
- wiki_bio_text: Value('string')
- gpt3_sentences: List(Value('string'))
- annotation: List(Value('string'))
- wiki_bio_test_idx: Value('int64')
- gpt3_text_samples: List(Value('string'))

[첫 번째 샘플]

■ gpt3_text | str
    John Russell Reynolds (1820–1876) was an English lawyer, judge, and author. He was born in London, the son of a barrister, and was educated at Eton College and Trinity College, Cambridge. He was called to the bar in 1845, and became a Queen's Counsel in 1859. He was appointed a judge of the Court of

■ wiki_bio_text | str
    Sir John Russell Reynolds, 1st Baronet (22 May 1828 – 29 May 1896) was a British neurologist and physician. Reynolds was born in Romsey, Hampshire, as the son of John Reynolds, an independent m

In [3]:
from collections import Counter

label_counter = Counter()
sample_count = Counter()
mismatch = 0

for row in data:
    label_counter.update(row["annotation"])
    sample_count[len(row["gpt3_text_samples"])] += 1
    if len(row["gpt3_sentences"]) != len(row["annotation"]):
        mismatch += 1

print("문서 수:", len(data))
print("전체 문장 수:", sum(label_counter.values()))
print("라벨 분포:", dict(label_counter))
print("문서당 샘플 수 분포:", dict(sample_count))
print("문장 수 != 라벨 수인 문서:", mismatch)

문서 수: 238
전체 문장 수: 1908
라벨 분포: {'major_inaccurate': 761, 'minor_inaccurate': 631, 'accurate': 516}
문서당 샘플 수 분포: {20: 238}
문장 수 != 라벨 수인 문서: 0


In [4]:
import re
import numpy as np

def tokenize(text):

    return re.findall(r"\w+", text.lower())


def unigram_scores(sentences, samples):
  avg_nll = []
  max_nll = []

  all_tokens = []

  for s in samples:
    words = tokenize(s)
    all_tokens.extend(words)

  counts = Counter(all_tokens)
  N =len(all_tokens)
  V =len(counts)


  for sentence in sentences:
    word_scores=[]
    for w in tokenize(sentence):
      p = (counts[w]+1)/(N+V+1)
      word_scores.append(-np.log(p))

    if len(word_scores) ==0:
      avg_nll.append(0.0)
      max_nll.append(0.0)

    else:
      avg_nll.append(np.mean(word_scores))
      max_nll.append(np.max(word_scores))

  return avg_nll, max_nll


# ---------- 테스트 ----------
_samples = ["the cat sat", "the cat ran"]
_sents = ["the cat", "the dog", ""]
_avg, _max = unigram_scores(_sents, _samples)

assert len(_avg) == 3 and len(_max) == 3
assert np.isclose(_avg[0], 1.299283, atol=1e-4)   # 샘플에 다 있는 문장
assert np.isclose(_max[0], 1.299283, atol=1e-4)
assert np.isclose(_avg[1], 1.848589, atol=1e-4)   # 'dog'는 샘플에 없음
assert np.isclose(_max[1], 2.397895, atol=1e-4)
assert _avg[2] == 0.0 and _max[2] == 0.0          # 빈 문장
assert _avg[1] > _avg[0]                          # 환각 의심 문장이 더 높아야 함
print("A 테스트 통과")

A 테스트 통과


In [5]:
from sklearn.metrics import precision_recall_curve, auc

def to_binary(annotations):
  ac_list = []
  for x in annotations:
    if x == "accurate":
      ac_list.append(0)

    else:
      ac_list.append(1)

  return ac_list


In [6]:
def compute_auc_pr(binary_labels, scores):
  precision, recall, thresholds = precision_recall_curve(binary_labels, scores)
  result = auc(recall, precision)
  return result


In [7]:
_labels = ['accurate', 'major_inaccurate', 'minor_inaccurate', 'accurate']
_bin = to_binary(_labels)

assert _bin == [0, 1, 1, 0]
assert np.isclose(compute_auc_pr(_bin, [0.1, 0.9, 0.8, 0.3]), 1.0)
assert np.isclose(compute_auc_pr(_bin, [0.9, 0.1, 0.8, 0.3]), 0.3333, atol=1e-3)
print("B 테스트 통과")

B 테스트 통과


In [8]:
all_labels, all_avg, all_max = [], [], []

for row in data:
    avg_s, max_s = unigram_scores(row["gpt3_sentences"], row["gpt3_text_samples"])
    all_avg.extend(avg_s)
    all_max.extend(max_s)
    all_labels.extend(to_binary(row["annotation"]))

print("문장 수:", len(all_labels))
print("환각 비율:", round(np.mean(all_labels), 4))   # 아무렇게나 찍을 때의 기준선
print("Unigram (avg) AUC-PR:", round(compute_auc_pr(all_labels, all_avg) * 100, 2))
print("Unigram (max) AUC-PR:", round(compute_auc_pr(all_labels, all_max) * 100, 2))

문장 수: 1908
환각 비율: 0.7296
Unigram (avg) AUC-PR: 83.58
Unigram (max) AUC-PR: 88.68


In [15]:
import torch, time, numpy as np
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli = SelfCheckNLI(device=device)   # potsawee/deberta-v3-large-mnli 자동 다운로드

t = time.time()
doc = data[0]
_ = nli.predict(sentences=doc["gpt3_sentences"], sampled_passages=doc["gpt3_text_samples"])
sec = time.time() - t
print(device, f"문서 1개 {sec:.1f}초 → 전체 예상 {sec*len(data)/60:.0f}분")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

SelfCheck-NLI initialized to device cpu


AttributeError: DebertaV2Tokenizer has no attribute batch_encode_plus

예시

In [9]:
samples = ["the cat sat", "the cat ran"]
all_tokens = []

for s in samples:               # 무엇에서 하나씩 꺼낼까?
    words = tokenize(s)         # 무엇을 단어로 자를까?
    all_tokens.extend(words)    # 무엇을 이어 붙일까?

print(all_tokens)

['the', 'cat', 'sat', 'the', 'cat', 'ran']


In [10]:
from collections import Counter

counts = Counter(all_tokens)      # 무엇을 세야 할까?
N = len(all_tokens)               # 전체 단어 수 (중복 포함)
V = len((counts))                 # 단어 종류 수

print(counts)
print("N =", N, "V =", V)

Counter({'the': 2, 'cat': 2, 'sat': 1, 'ran': 1})
N = 6 V = 4


In [11]:
import numpy as np

w= "the"
p= (counts[w]+1) / (N + V + 1)

In [12]:
score = -np.log(p)
print(w,p, score)

the 0.2727272727272727 1.2992829841302609


In [13]:
w = "dog"
p = (counts[w]+1) / (N + V + 1)
score = -np.log(p)
print(w,p, score)

dog 0.09090909090909091 2.3978952727983707


In [14]:
sentence = "the dog"
word_scores = []

for w in tokenize(sentence):
  p = (counts[w]+1)/(N+V+1)
  word_scores.append(-np.log(p))

print(word_scores)                             # 단어별 점수 2개
print("평균:", np.mean(word_scores))                   # 무엇의 평균?
print("최댓값:", np.max(word_scores))

[np.float64(1.2992829841302609), np.float64(2.3978952727983707)]
평균: 1.8485891284643157
최댓값: 2.3978952727983707
